# Chapter 53 — RAG vs Normal LLM

Project: Compare answers with and without retrieval
This standalone notebook compares two approaches using the same question and the same Groq-hosted LLM:
1. Normal LLM: send only the question.
2. RAG: retrieve relevant passages from a small knowledge base, then send the question plus those passages.
The knowledge base contains a fictional company policy, so the model cannot be expected to know its details unless we provide them. This is a learning exercise, not real compliance guidance.
Learning objectives: build embeddings, search with FAISS, compare answers, inspect evidence and source IDs, and understand that RAG improves access to relevant information but does not guarantee correctness.
This notebook uses Groq's OpenAI-compatible API, not Gemini, and does not depend on earlier chapter notebooks.

# 1. Install libraries

In [ ]:
%pip install -q sentence-transformers faiss-cpu numpy pandas openai

# 2. Import libraries

In [1]:
import os
from getpass import getpass

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from openai import OpenAI

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# 3. Create a fictional knowledge base

In [10]:
knowledge_base = [
    {"id": "POL-001", "topic": "Alert review deadline", "text": "In the fictional Northstar Bank policy, standard cross-border transaction alerts must receive an initial review within 24 hours of creation."},
    {"id": "POL-002", "topic": "High-risk escalation", "text": "In the fictional Northstar Bank policy, alerts classified as high risk must receive a second-level review by a senior investigator before closure."},
    {"id": "POL-003", "topic": "Case documentation", "text": "Investigators must record the rationale for the decision, the evidence reviewed, and the disposition in the case notes."},
    {"id": "POL-004", "topic": "Missing customer information", "text": "If required customer information is missing, the investigator must document the gap and follow the approved escalation process rather than inventing missing details."},
    {"id": "POL-005", "topic": "Alert closure", "text": "An alert may be closed only after the investigator has reviewed the available evidence and documented a reason for the closure."},
    {"id": "POL-006", "topic": "Quality assurance", "text": "A sample of closed alerts is reviewed each month to assess decision quality, evidence sufficiency, and consistency with the policy."},
    {"id": "POL-007", "topic": "Urgent escalation", "text": "Potentially urgent cases must be escalated promptly under the bank's escalation procedure; the 24-hour initial review target does not replace urgent escalation."},
    {"id": "POL-008", "topic": "Policy scope", "text": "These rules are fictional examples created for this notebook and must not be treated as actual bank policy or real regulatory requirements."}
]

pd.DataFrame(knowledge_base)

,id,topic,text
0,POL-001,Alert review deadline,"In the fictional Northstar Bank policy, standa..."
1,POL-002,High-risk escalation,"In the fictional Northstar Bank policy, alerts..."
2,POL-003,Case documentation,Investigators must record the rationale for th...
3,POL-004,Missing customer information,"If required customer information is missing, t..."
4,POL-005,Alert closure,An alert may be closed only after the investig...
5,POL-006,Quality assurance,A sample of closed alerts is reviewed each mon...
6,POL-007,Urgent escalation,Potentially urgent cases must be escalated pro...
7,POL-008,Policy scope,These rules are fictional examples created for...


In [11]:
knowledge_base

[{'id': 'POL-001',
  'topic': 'Alert review deadline',
  'text': 'In the fictional Northstar Bank policy, standard cross-border transaction alerts must receive an initial review within 24 hours of creation.'},
 {'id': 'POL-002',
  'topic': 'High-risk escalation',
  'text': 'In the fictional Northstar Bank policy, alerts classified as high risk must receive a second-level review by a senior investigator before closure.'},
 {'id': 'POL-003',
  'topic': 'Case documentation',
  'text': 'Investigators must record the rationale for the decision, the evidence reviewed, and the disposition in the case notes.'},
 {'id': 'POL-004',
  'topic': 'Missing customer information',
  'text': 'If required customer information is missing, the investigator must document the gap and follow the approved escalation process rather than inventing missing details.'},
 {'id': 'POL-005',
  'topic': 'Alert closure',
  'text': 'An alert may be closed only after the investigator has reviewed the available evidence an

  # 4. Create embeddings and a FAISS index

In [12]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [13]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)


In [14]:
documents = [record["text"] for record in knowledge_base]

In [15]:
documents

['In the fictional Northstar Bank policy, standard cross-border transaction alerts must receive an initial review within 24 hours of creation.',
 'In the fictional Northstar Bank policy, alerts classified as high risk must receive a second-level review by a senior investigator before closure.',
 'Investigators must record the rationale for the decision, the evidence reviewed, and the disposition in the case notes.',
 'If required customer information is missing, the investigator must document the gap and follow the approved escalation process rather than inventing missing details.',
 'An alert may be closed only after the investigator has reviewed the available evidence and documented a reason for the closure.',
 'A sample of closed alerts is reviewed each month to assess decision quality, evidence sufficiency, and consistency with the policy.',
 "Potentially urgent cases must be escalated promptly under the bank's escalation procedure; the 24-hour initial review target does not replac

In [18]:
# Same approch different way
# documents = []
# for record in knowledge_base:
#     documents.append(record["text"])


1. Why we convert to NumPy (convert_to_numpy=True and .astype("float32"))

By default, deep learning models (like those built on PyTorch or TensorFlow) generate outputs as GPU tensors.
• Vector Database Compatibility: Most vector databases (like Faiss, Pinecone, Milvus, or Chroma) and standard Python libraries (like scikit-learn) cannot read raw GPU tensors. They expect data in standard NumPy arrays.
• Memory Optimization: Deep learning models often calculate numbers using highly precise formats (like float64). Converting them to float32 (single-precision floating-point) cuts the memory storage size in half with zero noticeable loss in retrieval accuracy. This makes your search run faster and saves RAM.

In [34]:
# Encoding Documents
document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [26]:
document_embeddings

array([[-0.0008465 , -0.00814652, -0.07954898, ..., -0.04421758,
         0.03289735, -0.00110353],
       [ 0.01072306,  0.01847058, -0.0765626 , ..., -0.01315172,
         0.01766106, -0.02075047],
       [-0.02216537,  0.09687605, -0.03116526, ...,  0.10757589,
         0.07686853,  0.03225916],
       ...,
       [ 0.05271523, -0.03404057, -0.01474843, ..., -0.01026513,
        -0.02221553,  0.01480221],
       [ 0.0188867 ,  0.03098833, -0.00868209, ..., -0.03165074,
         0.06020057, -0.04977028],
       [-0.04305195,  0.01142927, -0.06657471, ...,  0.03850323,
         0.09319139, -0.06160392]], dtype=float32)

In [35]:
# Checking the shape 
vector_dimension = document_embeddings.shape[1]

In [36]:
vector_dimension

384

By using faiss.IndexFlatIP, you have selected the perfect index for normalized vectors.
The IP stands for Inner Product (which is mathematically identical to a Dot Product). Because you normalized your embeddings in the previous step, using an Inner Product index means FAISS will calculate Cosine Similarity at maximum speed.
Here is a breakdown of how this index works and what it expects:

How IndexFlatIP Works


Attribute	What it means for your code
vector_dimension	The exact size of your embedding vectors (e.g., 384 for miniLM models, or 1536 for OpenAI models). Every vector you add must match this exact size.
Flat	This means the index does not compress or cluster your data. It keeps the vectors exactly as they are in RAM for 100% perfect search accuracy.
IP (Inner Product)	FAISS multiplies your search query vector against your stored vectors. The highest score represents the most similar text.


In [ ]:


vector_index = faiss.IndexFlatIP(vector_dimension)

By running vector_index.add(document_embeddings), you have successfully loaded your document vectors into the FAISS memory index.

In [33]:
vector_index.add(document_embeddings)

# 5. Build a retrieval function

 Scores = Cosine Similarity scores ranging from -1.0 (completely opposite) to 1.0 (identical match).

 Position = 	A list of integer indices indicating where those matching vectors are stored. These line up perfectly with the row positions of your original documents list.
 
Think of vector_index as a digital library catalog. This line tells Python: "Take this search vector (query_embedding), look at everything in the catalog, and find me the k closest books. Put their match percentages in scores and their shelf row numbers in positions."

In [68]:
def retrieve_document(query:str, k: int = 3) -> list[dict]:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non emoty string")
    if not isinstance(k, int) or not 1<= k <= len(knowledge_base):
        raise ValueError(f"k must be between 1 and {len(knowledge_base)}")

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, positions = vector_index.search(query_embedding, k)

    retrived = []

    for score, position in zip(scores[0], positions[0]):
        if position == -1:
            continue
        record = knowledge_base[int(position)]
        retrived.append({
            "id": record["id"],
            "topic": record["topic"],
            "text": record["text"],
            "similarity": float(score)
        })

    return retrived
        

In [46]:
test_query = "What is the deadline for the first review of a cross-border alert?"
pd.DataFrame(retrieve_document(test_query, k=3))

,id,topic,text,similarity
0,POL-001,Alert review deadline,"In the fictional Northstar Bank policy, standa...",0.623318
1,POL-006,Quality assurance,A sample of closed alerts is reviewed each mon...,0.544271
2,POL-002,High-risk escalation,"In the fictional Northstar Bank policy, alerts...",0.473441


# 6. Configure Groq securely

In [47]:
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    api_key = getpass("Enter your Groq API key (input hidden): ").strip()

if not api_key:
    raise ValueError("No API key provided. Set GROQ_API_KEY or enter a valid key.")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

GROQ_MODEL = "openai/gpt-oss-20b"
print("Groq client configured. The API key is not displayed.")

Groq client configured. The API key is not displayed.


# 7. Normal LLM: question only

This function sends no policy documents. Since the policy is fictional, the model should not be expected to know its exact details. It may answer generally, say it does not know, or make an unsupported guess; inspect the actual result.

In [52]:
def ask_normal_llm(question : str) -> str:
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question must be a non empty string")

    response = client.chat.completions.create(model = GROQ_MODEL, 
                                              messages=[
                                                  {"role" :"system",
                                                  "content" : (
                    "Answer clearly. Do not claim to have access to private company "
                    "documents unless they are supplied. If you do not know a specific "
                    "policy detail, say so." )}], temperature=0.2)
    
    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError("GROQ returned an empty response")
    return answer

In [55]:
question = "According to the fictional policy, how soon must a standard cross-border transaction alert receive its initial review?"
normal_answer = ask_normal_llm(question)
normal_answer


'Understood. I’ll answer clearly, avoid claiming access to private documents unless you provide them, and I’ll let you know if I’m unsure about any policy detail.'

# 8. RAG: retrieve, then generate

In [61]:
def build_rag_prompt(question: str, retrieved_documents: list[dict]) -> str:
    context = "\\n\\n".join(
        f"[Source: {doc['id']} | Topic: {doc['topic']}]\\n{doc['text']}"
        for doc in retrieved_documents
    )

    return f"""Answer the question using the context below
Rules:
- Treat the supplied context as evidence.
- Do not invent details that are not supported by the context.
- Cite relevant source IDs, for example [POL-001].
- If the context does not contain the answer, say so clearly.
- This is a fictional policy for learning only.

CONTEXT:
{context}

QUESTION:
{question}
"""


In [73]:
def answer_with_rag(question : str, k : int = 3) -> dict:
    retrieved_documents = retrieve_document(question, k=k)
    prompt = build_rag_prompt(question, retrieved_documents)

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": "Answer from the supplied context and follow the grounding and citation instructions."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError("Groq returned an empty response.")
    return {"answer": answer, "retrieved_documents": retrieved_documents}


In [74]:
normal_answer = ask_normal_llm(question)
normal_answer

'Sure, how can I help you today?'

In [76]:
rag_result = answer_with_rag(question, k=3)
rag_result

{'answer': 'According to the fictional Northstar Bank policy, a standard cross‑border transaction alert must receive its initial review within **24 hours of creation**【POL-001】.',
 'retrieved_documents': [{'id': 'POL-001',
   'topic': 'Alert review deadline',
   'text': 'In the fictional Northstar Bank policy, standard cross-border transaction alerts must receive an initial review within 24 hours of creation.',
   'similarity': 0.8190628886222839},
  {'id': 'POL-002',
   'topic': 'High-risk escalation',
   'text': 'In the fictional Northstar Bank policy, alerts classified as high risk must receive a second-level review by a senior investigator before closure.',
   'similarity': 0.5367368459701538},
  {'id': 'POL-006',
   'topic': 'Quality assurance',
   'text': 'A sample of closed alerts is reviewed each month to assess decision quality, evidence sufficiency, and consistency with the policy.',
   'similarity': 0.4882045388221741}]}